# Longitudinal rCBV and radiation dose analysis

## 1. Introduction

This workflow estimates longitudinal changes in leakage-corrected relative cerebral blood volume and describes their spatial association with prior radiation dose. The May examination defines the reference grid. The October examination and planning CT are rigidly aligned to May anatomy, and RTDOSE is mapped through its verified reference frame.

Candidate changes and dose overlap support a research assessment of treatment-related change. Mixed injury and tumour, technical artifacts and other biological changes can produce similar findings. Clinical follow-up and other reference evidence are needed to interpret the findings.

The default run generates synthetic DICOM data with known spatial relationships. Patient runs use a local configuration and require documented registration review.

## 2. Configuration

The notebook searches for the checkout in the kernel's current directory and its ancestors. Set `NECROSIS_CONFIG` to the local patient YAML path before starting Jupyter to use patient data. Set `NECROSIS_OUTPUT` for a separate output location. The synthetic path creates a fresh run directory on each execution.

In [ ]:
import os
import sys
from pathlib import Path
from tempfile import mkdtemp

from IPython.display import Image, display
import matplotlib.pyplot as plt

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "run_analysis.py").is_file() and (path / "src").is_dir()),
    None,
)
if root is None:
    raise RuntimeError("Start Jupyter from the project root or its notebooks directory.")
sys.path.insert(0, str(root / "src"))

from necrosis_analysis.image_geometry import geometry_summary
from necrosis_analysis.pipeline import (
    analyze_inputs, array_image, export_results, inspect_inputs, load_config,
    make_figures, register_inputs,
)
from necrosis_analysis.synthetic import generate_synthetic_dataset
from necrosis_analysis.visualization import image_preview, make_viewer

configured_path = os.environ.get("NECROSIS_CONFIG")
output_override = os.environ.get("NECROSIS_OUTPUT")
if configured_path:
    config = load_config(Path(configured_path).expanduser().resolve())
    output_dir = (
        Path(output_override).expanduser().resolve() if output_override
        else root / "outputs" / "patient_notebook"
    )
else:
    output_root = (
        Path(output_override).expanduser().resolve() if output_override
        else root / "outputs" / "notebook"
    )
    output_root.mkdir(parents=True, exist_ok=True)
    run_dir = Path(mkdtemp(prefix="synthetic-", dir=output_root))
    config = load_config(None, overrides=generate_synthetic_dataset(run_dir / "dicom"))
    output_dir = run_dir / "analysis"

print(f"Mode = {config['mode']}")
print(f"Output directory = {output_dir}")

## 3. DICOM loading

Inspect the selected imaging series, study metadata, frame identifiers, native sampling and numerical scaling. Review anatomical candidates from the two MRI studies and resolve any ambiguous series in the local configuration.

In [ ]:
inputs = inspect_inputs(config, output_dir)
display(inputs.report)

for key, title in (
    ("may_anatomy", "May MRI anatomy"),
    ("october_anatomy", "October MRI anatomy"),
    ("may_rcbv", "May rCBV on its native grid"),
    ("october_rcbv", "October rCBV on its native grid"),
    ("ct", "Planning CT"),
):
    if key in inputs.images:
        figure = image_preview(inputs.images[key], title=title)
        display(figure)
        plt.close(figure)

## 4. Image geometry validation

DICOM patient coordinates are LPS (left, posterior, superior) in millimetres. NumPy volume indices are ordered z, y, x. The physical grid depends on spacing, origin and direction cosines. The two rCBV grids may differ in resolution and coverage even when their anatomy is aligned.

In [ ]:
display({key: geometry_summary(image) for key, image in inputs.images.items()})

## 5. Rigid registration

Register October anatomy and planning CT to the May anatomy. The mappings take May physical points to source physical points for resampling. Review within-session anatomy-to-rCBV mappings and compose the chain before sampling either rCBV map. Continuous values use linear interpolation, and label masks use nearest-neighbour interpolation.

In [ ]:
registered = register_inputs(config, inputs, output_dir)
display(geometry_summary(registered.reference))

## 6. Registration quality assurance

Inspect before/after fusion, checkerboards and boundaries in all three planes. Assess anatomical and perfusion coverage, the ventricles and the region under study. Inspect dose geometry on the planning CT and on the May grid. Use landmarks or independent masks when available.

The synthetic run compares estimated mappings against known geometry. For a patient run, use the saved review fingerprint to record a decision in the YAML with reviewer and rationale, then reload the notebook. Downstream analysis requires a matching approval.

In [ ]:
display(registered.report)
for path in sorted(registered.figures):
    print(path.name)
    display(Image(filename=str(path)))

## 7. rCBV normalization

Use a reviewed normal-appearing white matter ROI with valid perfusion at both dates. Exclude the skull, CSF, vessels, lesions and artefact from that reference. Time-specific median normalization can absorb a true global change. Justify the selected method from the source map scaling and reference-region measurements.

The shared analysis call enforces the registration QA decision and computes the results inspected in the following sections.

In [ ]:
analysis = analyze_inputs(config, inputs, registered, output_dir)
figures = make_figures(config, registered, analysis, output_dir)
figure_by_name = {path.stem: path for path in figures}
display(analysis.report["normalization"])
display(Image(filename=str(figure_by_name["normalization"])))

## 8. Longitudinal analysis

Signed change is October minus May in normalized rCBV units. The percent change requires a sufficiently large baseline value. The analysis retains common coverage and uses separate masks for changes in each direction. Threshold sweeps report the effect of the selected cutoff on candidate volumes.

In [ ]:
display(analysis.report["change"])
display(analysis.report["sensitivity"])
for name in ("longitudinal", "positive_change", "negative_change"):
    display(Image(filename=str(figure_by_name[name])))

## 9. Dose registration and interpretation

The source grid is scaled by `DoseGridScaling`, linked to the planning CT coordinate system and sampled into May anatomy. Resampling preserves the underlying dose scale. The EQD2 label requires documented evidence about the biological transformation and dose accumulation.

For patient interpretation, verify the alpha/beta ratio, source courses, accumulation method and treatment dates. Establish the treatment timing relative to both exams before drawing temporal conclusions.

In [ ]:
print(analysis.report["dose_label"])
display(registered.report["dose_provenance"])
display({"source": geometry_summary(inputs.images["dose"]),
         "May reference": geometry_summary(registered.images["dose"])})

## 10. Dose and rCBV spatial association

Calculate descriptive associations and dose-binned change summaries within common dose and perfusion coverage. Dose thresholds define nested regions within this analysis domain. Report the overlap denominators, and use candidate volumes in millilitres. Spatial autocorrelation limits what voxelwise associations can establish.

In [ ]:
display(analysis.report["dose"])
if "dose_association" in figure_by_name:
    display(Image(filename=str(figure_by_name["dose_association"])))

## 11. Candidate perfusion changes

Review the location of decreases and increases, and the regions where either change intersects elevated dose. Inspect the apparent change against anatomical findings and artefacts. Candidate labels describe the measured perfusion change and dose overlap.

In [ ]:
display(analysis.report["regions"])
display(Image(filename=str(figure_by_name["candidate_overlay"])))

## 12. Interactive viewer

Use the plane and slice controls to navigate all views on the same grid. Select the baseline, follow-up or change layer and adjust transparency and thresholds. Indices and physical coordinates refer to the displayed reference sampling. Set `NECROSIS_STATIC_VIEWER=1` for a static display when running without a widget frontend.

In [ ]:
reference = registered.reference
change_threshold = config["analysis"]["absolute"]
viewer = make_viewer(
    reference,
    array_image(analysis.maps["may_normalized"], reference),
    array_image(analysis.maps["october_normalized"], reference),
    array_image(analysis.maps["signed"], reference),
    dose=registered.images["dose"],
    dose_label=analysis.report["dose_label"],
    ct=registered.images["ct"],
    percent=array_image(analysis.maps["percent"], reference),
    change_threshold=change_threshold if change_threshold is not None else 0.25,
    dose_threshold=config["analysis"]["overlay_dose_threshold"],
    interactive=os.environ.get("NECROSIS_STATIC_VIEWER") != "1",
)
display(viewer)

## 13. Quantitative report

Review common imaging coverage, registration quality, normalization factors, change and dose summaries. Distinguish uncertainty in dose accumulation and timing from uncertainty in image registration and rCBV comparability.

In [ ]:
display(analysis.report)

## 14. Export results

Write registered images, masks and quantitative maps, QA and publication figures, CSV/JSON tables and the readable report into the output tree.

In [ ]:
report_path = export_results(config, inputs, registered, analysis, output_dir, figures=figures)
print(f"Report = {report_path}")
for path in sorted(output_dir.iterdir()):
    print(path.name)